# Cocogoat 2026 - Task 2.1: surprise-gated long-term memory for the world model (A/B test)

Two world models on the frozen Task 2 VAE, identical except for one thing: `ltm` adds a bank of past frame latents chosen by surprise (prediction error), read by cross-attention - the long-term memory of your ViVAE notes. `base` is the Task 2 model (EMA memory) trained the same way: 64-frame clips, chunks of 16 frames, rotary time positions. Moving MNIST is the smoke test; Shelter is the main experiment (clips cross scene cuts there). Needs the Task 2 VAEs (`runs/task2/<dataset>/vae/vae.pt`), otherwise one is trained first. Outputs: `MyDrive/Cocogoat/runs/task2_1/<dataset>/` (`compare.md`, `compare.png`, `<arm>/samples/`).

**How to run:** `Runtime -> Change runtime type -> G4 (RTX PRO 6000)`, set the budgets in the settings cell (5),
then `Runtime -> Run all`.
Keep this tab open (Colab Pro stops idle/closed sessions). Everything is written to
`MyDrive/Cocogoat/runs/...`; if Colab disconnects, just *Run all* again - training resumes from the last checkpoint.
You do not need to inspect results: metrics, sample grids and logs are saved to Drive for analysis.

In [ ]:
# 1) Mount Google Drive (outputs + datasets live there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Get / update the code (public repo, branch claude/cocogoat-revival-glfnmz)
import os, subprocess
REPO, BRANCH = "https://github.com/Martyn0324/Cocogoat.git", "claude/cocogoat-revival-glfnmz"
if not os.path.exists('/content/Cocogoat/.git'):
    subprocess.run(['git', 'clone', '-b', BRANCH, '--depth', '1', REPO, '/content/Cocogoat'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Cocogoat', 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/Cocogoat', 'reset', '--hard', 'FETCH_HEAD'], check=True)
print(subprocess.run(['git', '-C', '/content/Cocogoat', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
os.chdir('/content/Cocogoat')

In [ ]:
# 3) Extra packages (torch/torchvision/transformers come with Colab)
!pip -q install -r /content/Cocogoat/requirements-colab.txt
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
# Colab Secrets are only readable from the notebook kernel: export HF_TOKEN for the training subprocesses
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN exported')
except Exception as exc:
    print('No HF_TOKEN secret (only needed for gated Hugging Face datasets):', type(exc).__name__)
os.environ['PYTHONUNBUFFERED'] = '1'

In [ ]:
# 4) Smoke test (tiny models, a few steps, both arms, always from scratch)
!python -m cocogoat.tasks.task2_1_ltm --smoke --dataset mmnist --run_root /content/drive/MyDrive/Cocogoat/runs/task2_1_smoke

In [ ]:
# 5) Settings - edit freely. WM_HOURS = training budget PER ARM (two arms per dataset).
WM_HOURS = 0.75
RUN_ROOT = '/content/drive/MyDrive/Cocogoat/runs/task2_1'
TASK2_ROOT = '/content/drive/MyDrive/Cocogoat/runs/task2'   # frozen VAEs are taken from here
LTM_SLOTS = 8                       # frames kept in the long-term memory
EVAL_CONTEXT, EVAL_HORIZON = 64, 64  # real frames shown, then frames generated
SHELTER_START_FRAME = 500           # must match the Task 2 run whose VAE is reused
GIF_CLIPS, GIF_FPS = 4, 8
COMMON = (f'--wm_hours {WM_HOURS} --run_root {RUN_ROOT} --ltm_slots {LTM_SLOTS} --eval_context {EVAL_CONTEXT} '
          f'--eval_horizon {EVAL_HORIZON} --gif_clips {GIF_CLIPS} --gif_fps {GIF_FPS}')

In [ ]:
# 6) Moving MNIST (smoke-test dataset): base arm, then ltm arm, then the comparison
!python -m cocogoat.tasks.task2_1_ltm --dataset mmnist {COMMON} --vae_from {TASK2_ROOT}/mmnist/vae/vae.pt

In [ ]:
# 7) Shelter (main experiment)
!python -m cocogoat.tasks.task2_1_ltm --dataset shelter {COMMON} --vae_from {TASK2_ROOT}/shelter/vae/vae.pt --start_frame {SHELTER_START_FRAME}

In [ ]:
# 8) Results: table, PSNR/LPIPS-vs-horizon chart, rollout GIFs of both arms
from IPython.display import Image, Markdown, display
for ds in ['mmnist', 'shelter']:
    run = f'{RUN_ROOT}/{ds}'
    try:
        display(Markdown(open(f'{run}/compare.md').read()))
        display(Image(f'{run}/compare.png'))
        for arm in ['base', 'ltm']:
            display(Markdown(f'**{ds} / {arm}**'), Image(f'{run}/{arm}/samples/rollout_final.gif'))
    except FileNotFoundError as exc:
        print(ds, 'not finished:', exc)